# Archived SD1.5 Prompt-Mismatched In-Range CFG Ablation

This notebook preserves visualization of the original five-rate CFG ablation with two trials for CFG 1, 3, and 5. It reads only archived `_old` result paths and does not synchronize data from the corrected study.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (search_root / 'ktilde' / 'config.json').is_file() and (search_root / 'src').is_dir():
        SD15_ROOT = search_root
        break
else:
    raise FileNotFoundError('Could not find sd1.5 project root.')

os.environ.setdefault('MPLCONFIGDIR', str(SD15_ROOT / 'results' / 'analysis' / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

analysis_dir = SD15_ROOT / 'analyze_results'
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import sd15_cfg_ablation_analysis as cfgviz
importlib.reload(cfgviz)

EXPERIMENT = 'prompt_mismatched_in_range_old'
OUTPUT_DIR = SD15_ROOT / 'results' / 'analysis' / 'ablation' / 'prompt_mismatched_old' / 'sunset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SD15_ROOT, OUTPUT_DIR

## Load Archived Results

The archived experiment key prevents reference synchronization and loads the preserved five-rate result tree.

In [ ]:
sync_report = cfgviz.sync_main_references(SD15_ROOT, experiment=EXPERIMENT)
display(sync_report.groupby(['distribution_key', 'status'], dropna=False).size().reset_index(name='copy_events'))

rows = cfgviz.load_cfg_ablation_rows(SD15_ROOT, experiment=EXPERIMENT)
print(f'Loaded {len(rows)} reconstruction rows')
display(cfgviz.count_table(rows))
display(rows[['distribution_key', 'line_condition', 'samp_perc', 'repeat_id', 'psnr_db', 'ssim', 'pixel_mae', 'case_root']].head())

## Metric Curves

This cell plots PSNR and SSIM versus sampling ratio for each sampling distribution in the ablation. Within each subplot, the colored lines compare recovery settings: unconditioned, CFG 1, CFG 3, CFG 5, and CFG 7.5. The dashed black line is the zero-filled inverse FFT baseline when it is available.

The helper writes PDF outputs into `OUTPUT_DIR`. With `band='ci'`, the shaded bands are 95% normal-approximation confidence intervals over the repeats that have finished.

In [ ]:
metric_outputs = cfgviz.plot_metric_curves(rows, output_dir=OUTPUT_DIR, show=True, band='ci')
metric_outputs

## Reconstruction Panels

This cell makes an image panel at one sampling ratio, `0.00125` by default. Each row is a sampling prior, and the columns show ground truth, zero-filled baseline, and the best available reconstruction for each recovery setting.

The best reconstruction in each tile is selected by PSNR first and SSIM second. Set `samp_perc=None` in the code to use the largest sampling ratio shared by the loaded panel columns instead of forcing the low-ratio view.

In [ ]:
# Shows a fixed low sampling ratio for the image panel. Set samp_perc=None to use the largest ratio shared by available panel columns.
panel_outputs = cfgviz.plot_reconstruction_panel(
    rows,
    samp_perc = 0.00125,
    sd15_root=SD15_ROOT,
    output_dir=OUTPUT_DIR,
    show=True,
)
panel_outputs